<a href="https://colab.research.google.com/github/vedant16052006/Data_Science_lab/blob/main/PythonPrograms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BFS & DFS


## BFS & DFS Demonstration

First, let's define a simple graph using an adjacency list representation. This graph will be used for both BFS and DFS demonstrations.

In [1]:
# Define a graph using an adjacency list
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("Graph definition (Adjacency List):")
for node, neighbors in graph.items():
    print(f"  {node}: {neighbors}")

Graph definition (Adjacency List):
  A: ['B', 'C']
  B: ['A', 'D', 'E']
  C: ['A', 'F']
  D: ['B']
  E: ['B', 'F']
  F: ['C', 'E']


### Breadth-First Search (BFS)

BFS explores all the neighbor nodes at the present depth prior to moving on to nodes at the next depth level. It typically uses a queue data structure.

In [2]:
from collections import deque

def bfs(graph, start_node):
    visited = set() # To keep track of visited nodes
    queue = deque([start_node]) # Initialize a queue with the starting node
    traversal_order = []

    visited.add(start_node)

    while queue:
        current_node = queue.popleft() # Dequeue a node
        traversal_order.append(current_node)

        # Explore neighbors
        for neighbor in graph[current_node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor) # Enqueue unvisited neighbors
    return traversal_order

print("BFS Traversal starting from node 'A':")
bfs_result = bfs(graph, 'A')
print(bfs_result)

BFS Traversal starting from node 'A':
['A', 'B', 'C', 'D', 'E', 'F']


### Depth-First Search (DFS)

DFS explores as far as possible along each branch before backtracking. It typically uses a stack data structure (or recursion, which uses the call stack).

In [3]:
def dfs(graph, start_node):
    visited = set() # To keep track of visited nodes
    stack = [start_node] # Initialize a stack with the starting node
    traversal_order = []

    while stack:
        current_node = stack.pop() # Pop a node from the stack

        if current_node not in visited:
            visited.add(current_node)
            traversal_order.append(current_node)

            # Push unvisited neighbors onto the stack
            # Note: For consistent order, push in reverse of desired exploration order
            # For example, if 'A' has neighbors ['B', 'C'], pushing 'C' then 'B'
            # will make 'B' explored first. If pushing 'B' then 'C', 'C' is explored first.
            # Here, we'll iterate through sorted neighbors to ensure a consistent output.
            for neighbor in sorted(graph[current_node], reverse=True):
                if neighbor not in visited:
                    stack.append(neighbor)
    return traversal_order

print("DFS Traversal starting from node 'A':")
dfs_result = dfs(graph, 'A')
print(dfs_result)

DFS Traversal starting from node 'A':
['A', 'B', 'D', 'E', 'F', 'C']


# A* algorithm.

The A* search algorithm is a popular pathfinding algorithm. It is an informed search algorithm, meaning it uses heuristic knowledge to guide its search. A* works by finding the path with the lowest cost from a starting node to a target node.

The cost function for A* is typically defined as: `f(n) = g(n) + h(n)`
*   `g(n)`: The cost to get from the start node to node `n`.
*   `h(n)`: The estimated cost (heuristic) to get from node `n` to the goal node.
*   `f(n)`: The total estimated cost of the path through node `n` to the goal.

To implement A*, we'll typically use a priority queue to keep track of the nodes to visit, ordered by their `f(n)` value.

In [4]:
import heapq # For implementing a priority queue

def a_star_search(graph, start, goal, heuristic):
    # The set of discovered nodes that may need to be (re-)expanded.
    # Initially, only the start node is known.
    # This is a min-heap, storing (f_score, node)
    open_set = [(heuristic[start], start)]

    # For node n, came_from[n] is the node immediately preceding it on the cheapest path from start
    # to n currently known.
    came_from = {}

    # For node n, g_score[n] is the cost of the cheapest path from start to n currently known.
    g_score = {node: float('inf') for node in graph}
    g_score[start] = 0

    # For node n, f_score[n] = g_score[n] + heuristic[n]. f_score[n] represents our current best guess
    # as to how cheap a path from start to finish can be if it goes through n.
    f_score = {node: float('inf') for node in graph}
    f_score[start] = heuristic[start] # For start node, f_score is just its heuristic

    while open_set:
        # Current node is the node in open_set with the lowest f_score
        current_f_score, current_node = heapq.heappop(open_set)

        if current_node == goal:
            # Reconstruct path if we reached the goal
            path = []
            while current_node in came_from:
                path.append(current_node)
                current_node = came_from[current_node]
            path.append(start) # Add the start node
            return path[::-1] # Reverse to get path from start to goal

        for neighbor, weight in graph[current_node].items():
            # d(current, neighbor) is the weight of the edge from current to neighbor
            # tentative_g_score is the distance from start to the neighbor through current
            tentative_g_score = g_score[current_node] + weight

            if tentative_g_score < g_score[neighbor]:
                # This path to neighbor is better than any previous one. Record it!
                came_from[neighbor] = current_node
                g_score[neighbor] = tentative_g_score
                f_score[neighbor] = g_score[neighbor] + heuristic[neighbor]
                heapq.heappush(open_set, (f_score[neighbor], neighbor))

    return None # No path found


Let's define a sample graph with edge weights and a heuristic function for each node to demonstrate the A* algorithm.

In [5]:
# Define a graph with weights (Adjacency list with dictionaries for neighbors and weights)
weighted_graph = {
    'A': {'B': 1, 'C': 3},
    'B': {'A': 1, 'D': 4, 'E': 2},
    'C': {'A': 3, 'F': 5},
    'D': {'B': 4, 'G': 1},
    'E': {'B': 2, 'F': 1, 'G': 3},
    'F': {'C': 5, 'E': 1, 'G': 2},
    'G': {'D': 1, 'E': 3, 'F': 2}
}

# Define heuristic values (estimated distance from each node to the goal 'G')
heuristic = {
    'A': 7,
    'B': 6,
    'C': 4,
    'D': 2,
    'E': 3,
    'F': 2,
    'G': 0  # Goal node has a heuristic of 0
}

print("Weighted Graph:")
for node, neighbors in weighted_graph.items():
    print(f"  {node}: {neighbors}")

print("\nHeuristic Values (to G):")
for node, h_val in heuristic.items():
    print(f"  {node}: {h_val}")


Weighted Graph:
  A: {'B': 1, 'C': 3}
  B: {'A': 1, 'D': 4, 'E': 2}
  C: {'A': 3, 'F': 5}
  D: {'B': 4, 'G': 1}
  E: {'B': 2, 'F': 1, 'G': 3}
  F: {'C': 5, 'E': 1, 'G': 2}
  G: {'D': 1, 'E': 3, 'F': 2}

Heuristic Values (to G):
  A: 7
  B: 6
  C: 4
  D: 2
  E: 3
  F: 2
  G: 0


In [6]:
start_node = 'A'
goal_node = 'G'

print(f"Finding path from {start_node} to {goal_node} using A* algorithm:")
path = a_star_search(weighted_graph, start_node, goal_node, heuristic)

if path:
    print(f"Shortest path: {path}")
    # Calculate the total cost of the path
    total_cost = 0
    for i in range(len(path) - 1):
        current = path[i]
        next_node = path[i+1]
        total_cost += weighted_graph[current][next_node]
    print(f"Total cost of path: {total_cost}")
else:
    print("No path found.")


Finding path from A to G using A* algorithm:
Shortest path: ['A', 'B', 'E', 'G']
Total cost of path: 6


# Minimax Algorithm

# Minimax Algorithm

The Minimax algorithm is a decision-making algorithm used in artificial intelligence, game theory, and other fields. It is primarily used for two-player turn-based games (like Tic-Tac-Toe, Chess, Go) where both players try to maximize their own score and minimize the opponent's score.

**Key Concepts:**
*   **Players:** A 'Maximizer' (who wants to achieve the highest possible score) and a 'Minimizer' (who wants to achieve the lowest possible score for the Maximizer).
*   **Game Tree:** A tree structure where each node represents a state of the game, and edges represent possible moves.
*   **Terminal States:** Nodes at the end of the game tree (leaves) where the game ends, and a utility value (score) is assigned.
*   **Recursion:** The algorithm works recursively, exploring all possible moves to a certain depth.

**How it works:**
1.  **Evaluation:** At the terminal nodes, a static evaluation function assigns a numerical score.
2.  **Maximizing Player's Turn:** If it's the Maximizer's turn, they will choose the move that leads to the state with the maximum possible score (assuming the Minimizer plays optimally).
3.  **Minimizing Player's Turn:** If it's the Minimizer's turn, they will choose the move that leads to the state with the minimum possible score for the Maximizer (assuming the Maximizer plays optimally).
4.  **Backtracking:** The scores are propagated up the game tree until the root node, determining the optimal move for the current player.

In [7]:
def minimax(node, depth, maximizing_player):
    # This simple example assumes 'node' is a dictionary representing the game tree.
    # Terminal nodes are those without 'children' or when depth is 0.
    # The 'value' key holds the utility for terminal nodes.

    if depth == 0 or 'children' not in node or not node['children']:
        # If it's a terminal node or max depth is reached, return its value
        return node['value']

    if maximizing_player:
        max_eval = float('-inf')
        for child_key in node['children']:
            child_node = node['children'][child_key]
            # Recursive call for the next level, switching to minimizing player
            evaluation = minimax(child_node, depth - 1, False)
            max_eval = max(max_eval, evaluation)
        return max_eval
    else:
        min_eval = float('inf')
        for child_key in node['children']:
            child_node = node['children'][child_key]
            # Recursive call for the next level, switching to maximizing player
            evaluation = minimax(child_node, depth - 1, True)
            min_eval = min(min_eval, evaluation)
        return min_eval

print("Minimax algorithm defined.")

Minimax algorithm defined.


Let's create a simple game tree to demonstrate the Minimax algorithm. The values at the leaf nodes represent the utility for the maximizing player.

In [8]:
# Example Game Tree
# Max player wants to maximize the score, Min player wants to minimize it (for Max).
# Leaf nodes have 'value' key.

game_tree = {
    'name': 'Root (Max)',
    'children': {
        'A': {
            'name': 'A (Min)',
            'children': {
                'A1': {'name': 'A1 (Max)', 'children': {
                    'A1a': {'name': 'A1a', 'value': 3},
                    'A1b': {'name': 'A1b', 'value': 5}
                }},
                'A2': {'name': 'A2 (Max)', 'children': {
                    'A2a': {'name': 'A2a', 'value': 2},
                    'A2b': {'name': 'A2b', 'value': 9}
                }}
            }
        },
        'B': {
            'name': 'B (Min)',
            'children': {
                'B1': {'name': 'B1 (Max)', 'children': {
                    'B1a': {'name': 'B1a', 'value': 1},
                    'B1b': {'name': 'B1b', 'value': 2}
                }},
                'B2': {'name': 'B2 (Max)', 'children': {
                    'B2a': {'name': 'B2a', 'value': 0},
                    'B2b': {'name': 'B2b', 'value': 1}
                }}
            }
        }
    }
}

print("Game tree defined for demonstration.")

Game tree defined for demonstration.


In [9]:
max_depth = 4 # The depth of the game tree from the root to the leaf nodes
start_maximizing_player = True # The root is a maximizing player's turn

print(f"Starting Minimax search from the root node with a depth of {max_depth} (assuming Maximizing Player starts)...")
optimal_value = minimax(game_tree, max_depth, start_maximizing_player)

print(f"The optimal value for the maximizing player from the root is: {optimal_value}")

Starting Minimax search from the root node with a depth of 4 (assuming Maximizing Player starts)...
The optimal value for the maximizing player from the root is: 5
